# LGN Converter — URL Stream to Timestamped Transcript

This notebook runs the first prototype stage on an authorized 1–2 minute source. It consumes the remote stream without saving the original video. A temporary mono WAV is created for Whisper.

## 1. Confirm the GPU
Before running this cell, select **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU is not enabled"
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))


## 2. Install the project

In [ ]:
![ -d /content/lngconverter/.git ] || git clone https://github.com/cubatic/lngconverter.git /content/lngconverter
%cd /content/lngconverter
!git pull --ff-only
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q -e '.[inference]'
print("Installation complete")


## 3. Configure the prototype
YouTube mode must only be enabled for a source for which you have the required content rights and platform authorization. No password, cookie, or token is required by this notebook.

In [ ]:
import os

os.environ["LGN_ENABLE_YOUTUBE_SOURCE"] = "true"
os.environ["LGN_WHISPER_MODEL"] = "small"
os.environ["LGN_WHISPER_DEVICE"] = "cuda"
os.environ["LGN_WHISPER_COMPUTE_TYPE"] = "float16"
os.environ["LGN_MAX_SOURCE_SECONDS"] = "180"
os.environ["LGN_KEEP_AUDIO_ARTIFACTS"] = "false"

print("Configuration ready")


## 4. Start transcription
Use `zh` for Chinese, `ko` for Korean, or `en` for English. Leave it blank to let Whisper detect the language.

In [ ]:
import time
from getpass import getpass

from lgnconverter.config import Settings
from lgnconverter.jobs import JobStore
from lgnconverter.models import CreateJobRequest

source_url = getpass("Authorized source URL: ").strip()
source_language = input("Source language (zh/ko/en, blank=auto): " ).strip() or None
clip_start = float(input("Clip start in seconds (blank=0): " ).strip() or 0)
clip_duration = float(input("Clip duration in seconds (blank=120, max=180): " ).strip() or 120)
assert source_url, "A source URL is required"

store = JobStore(Settings())
job = store.create(CreateJobRequest(
    source_url=source_url,
    authorization_confirmed=True,
    source_language=source_language,
    target_language="hi",
    clip_start_seconds=clip_start,
    clip_duration_seconds=clip_duration,
))
print("Job ID:", job.id)

last_status = None
while True:
    result = store.get(job.id)
    if result.status != last_status:
        print("Status:", result.status.value)
        last_status = result.status
    if result.status.value in {"completed", "failed"}:
        break
    time.sleep(3)

if result.status.value == "failed":
    raise RuntimeError(result.error)

print(f"Detected language: {result.detected_language}")
print(f"Processed duration: {result.duration_seconds:.2f}s")
print(f"Segments: {len(result.transcript)}")
for segment in result.transcript:
    print(f"[{segment.start:7.2f} → {segment.end:7.2f}] {segment.text}")
